## **<mark>Import the necessary Packages</mark>**

In [1]:
from datetime import datetime
import requests, pandas, uuid
from pyspark.sql.functions import current_timestamp, lit, col
from notebookutils import mssparkutils
from pyspark.sql.types import *
from datetime import datetime

StatementMeta(, 45c46d5b-86b5-4c9e-bc52-4feef0120463, 3, Finished, Available, Finished, False)

In [2]:
# here i am using a toggle parameter feature its alow to give a value for a parameter from a pipeline
end_date = datetime.today().strftime("%Y-%m-%d")
start_date =datetime.today().strftime("%Y-%m-%d")

# this lat and long located in chennai
latitude = 13.0827
longitude = 80.2707

# Pipelne run id data for API_Audit_Table and Notebook_Audit_Table from Base Parameter
pipeline_run_id = ""

StatementMeta(, 45c46d5b-86b5-4c9e-bc52-4feef0120463, 4, Finished, Available, Finished, False)

## **<mark>Null and Empty  value handle for base parameters from pipeline </mark>**

In [3]:
# Start Date
if str(start_date).strip().lower() in ["", "null", "none"]:
    start_date = datetime.today().strftime("%Y-%m-%d")

# End Date
if str(end_date).strip().lower() in ["", "null", "none"]:
    end_date = datetime.today().strftime("%Y-%m-%d")

# Latitude
if str(latitude).strip().lower() in ["", "null", "none"]:
    latitude = 13.0827  # chennai latitude
else:
    latitude = float(latitude)

# Longitude
if str(longitude).strip().lower() in ["", "null", "none"]:
    longitude = 80.2707  # chennai longitude
else:
    longitude = float(longitude)


StatementMeta(, 45c46d5b-86b5-4c9e-bc52-4feef0120463, 5, Finished, Available, Finished, False)

## **<mark>Creating Audit Variable </mark>**

In [4]:
run_id = str(uuid.uuid4())
notebook_name = "Bronze Notebook"
start_time = datetime.now()
status = "none"
records_fetched = 0
records_processed = 0
error_message = "none"

StatementMeta(, 45c46d5b-86b5-4c9e-bc52-4feef0120463, 6, Finished, Available, Finished, False)

In [5]:

try:

    #=====================================================
    # API URL
    #=====================================================

    url = f"https://archive-api.open-meteo.com/v1/era5?latitude={latitude}&longitude={longitude}&start_date={start_date}&end_date={end_date}&hourly=temperature_2m"
    #=====================================================
    # API Request
    #=====================================================

    response = requests.get(url)

    response.raise_for_status()

    data = response.json()

    #=====================================================
    # Convert JSON to Pandas DataFrame
    #=====================================================

    pdf = pandas.DataFrame({
        "Datetime": data["hourly"]["time"],
        "Temperature": data["hourly"]["temperature_2m"],
        "latitude": data["latitude"],
        "longitude": data["longitude"],
        "generationtime_ms": data["generationtime_ms"],
        "utc_offset_seconds": data["utc_offset_seconds"],
        "timezone": data["timezone"],
        "elevation": data["elevation"],
        "time_type": data["hourly_units"]["time"],
        "temp_type": data["hourly_units"]["temperature_2m"]
    })

    #=====================================================
    # Pandas -> Spark
    #=====================================================

    df = spark.createDataFrame(pdf)

    #=====================================================
    # Add Metadata
    #=====================================================

    file_name = f"weather_raw_{end_date}.json"

    df = df.withColumn("ingestion_time", current_timestamp())\
            .withColumn("source_file", lit(file_name))
    

    records_processed = df.count()

    #=====================================================
    # Save Bronze Data
    #=====================================================

    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bronze.weather_data")

    #=====================================================
    # status for audit API audit table
    #=====================================================
    
    status = "Success"

except Exception as e:

    status = "Failed"

    error_message = str(e)

    raise

finally:

    end_time = datetime.now()

    duration = (end_time - start_time).total_seconds()

    #=====================================================
    # API AUDIT
    #=====================================================

    api_audit_df = spark.createDataFrame([{
        "RunID": run_id,
        "PipelineRunID": pipeline_run_id,
        "APIName": "Open Meteo",
        "API_URL": url if 'url' in locals() else "",
        "Status": status,
        "RecordsFetched": int(records_fetched),
        "StartTime": start_time,
        "EndTime": end_time,
        "DurationSeconds": float(duration),
        "ErrorMessage": error_message,
        "CreatedOn": datetime.now()}])

    api_audit_df = api_audit_df.withColumn("RecordsFetched", col("RecordsFetched").cast("int"))\
    .withColumn("DurationSeconds", col("DurationSeconds").cast("double"))
    
    #=====================================================
    # Save audit Data
    #=====================================================

    spark.sql("CREATE SCHEMA IF NOT EXISTS audit")

    api_audit_df.write.mode("append").saveAsTable("audit.api_audit")

    #=====================================================
    # NOTEBOOK AUDIT
    #=====================================================

    notebook_audit_df = spark.createDataFrame([{
        "RunID": str(uuid.uuid4()),
        "PipelineRunID": pipeline_run_id,
        "NotebookName": notebook_name,
        "Status": status,
        "RecordsProcessed": int(records_processed),
        "StartTime": start_time,
        "EndTime": end_time,
        "DurationSeconds": float(duration),
        "ErrorMessage": error_message,
        "CreatedOn": datetime.now()}])

    #=====================================================
    # Save audit Data
    #=====================================================
    notebook_audit_df = notebook_audit_df.withColumn("RecordsProcessed", col("RecordsProcessed").cast("bigint"))
    notebook_audit_df = notebook_audit_df.withColumn("DurationSeconds", col("DurationSeconds").cast("double"))

    

    notebook_audit_df.write.mode("append").saveAsTable("audit.notebook_audit")

StatementMeta(, 45c46d5b-86b5-4c9e-bc52-4feef0120463, 7, Finished, Available, Finished, False)